In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus(2)

import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import scanpy as sc
import optax
from tqdm import tqdm

from sde import SDE
from utils import compute_kds, median_bandwidth
from dataloader import PerturbationDataLoader

In [ ]:
def train(
    adata: sc.AnnData,
    n_steps: int = 10000,
    batch_size: int = 64,
    lr: float = 1e-3,
    seed: int = 0,
    log_every: int = 50,
    bw_sample_size: int = 2000,
):
    """Minimize the KDS loss over interventional minibatches.

    Parameters
    ----------
    adata          : AnnData with .X and .obs["gene"]
    n_steps        : number of gradient update steps
    batch_size     : observations per minibatch
    lr             : Adam learning rate
    seed           : JAX PRNGKey seed for parameter initialisation
    log_every      : print loss every this many steps
    bw_sample_size : number of rows sampled from the full dataset to
                     compute the global median-heuristic bandwidth (computed
                     once before training, not per minibatch)

    Returns
    -------
    params : dict   final Flax parameter dict
    losses : list   scalar KDS loss recorded at each step
    """
    d = adata.n_vars
    sde = SDE(n_vars=d)
    loader = PerturbationDataLoader(adata, batch_size=batch_size, seed=seed)
    optimizer = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adam(lr),
    )

    key = jax.random.PRNGKey(seed)
    dummy_x = jnp.zeros((batch_size, d))
    params = sde.init(key, dummy_x, None, method=sde.forward)["params"]
    opt_state = optimizer.init(params)

    # Compute bandwidth once on a large sample of the full dataset.
    X_bw = loader.sample_dataset(bw_sample_size)
    bw = median_bandwidth(X_bw)
    print(f"Global bandwidth (n={bw_sample_size}): {float(bw):.4f}")

    def loss_fn(params, X, u):
        return compute_kds(sde, params, X, u=u, bandwidth=bw)

    losses = []
    for step_idx in tqdm(range(n_steps)):
        u, Xu = loader.sample()

        loss, grads = jax.value_and_grad(loss_fn)(params, Xu, u)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)

        # SDE.param_stats(params)

        losses.append(float(loss))
        if step_idx % log_every == 0:
            print(f"step {step_idx:4d} | KDS = {loss:.6f}")

    return params, losses




In [ ]:
adata = sc.read_h5ad(
    "/ewsc/pboyeau/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.X.toarray()

# surviving_genes = (adata.obs["gene"].value_counts() >= 20) | adata.obs["gene"].str.startswith("Control")
# surviving_genes = surviving_genes[surviving_genes].index
# adata = adata[:, adata.var.index.isin(surviving_genes)].copy()

In [ ]:
print(adata)
print(adata.obs["gene"].value_counts().sort_index().head(10), "\n...")

print("\n=== Training ===")
params, losses = train(adata, n_steps=500, batch_size=5, lr=1e-3)

print(f"\nInitial KDS : {losses[0]:.6f}")
print(f"Final   KDS : {losses[-1]:.6f}")

In [ ]:
plt.plot(losses)

In [ ]:
plt.plot(losses)